In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from scipy.stats import norm
import os

from model import *
from data import generate_gmm_data, generate_uos_data, generate_random_data
from train_utils import train

from plot import *
from utils import *


In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
relu = lambda X: np.maximum(X, 0)
d_relu = lambda X: np.maximum(np.sign(X), 0)

elu = lambda X, alpha=1.0: ( alpha * (np.exp(X) - 1) ) * (X <= 0) + (X * (X > 0))
d_elu = lambda X, alpha=1.0: ( alpha * np.exp(X) ) * (X <= 0) + (X > 0)

gelu = lambda X: X * norm.cdf(X)
d_gelu = lambda X: norm.cdf(X) + X * norm.pdf(X)

sigmoid = lambda X: 1 / (1 + np.exp(-X))
d_sigmoid = lambda X: sigmoid(X) * (1 - sigmoid(X))

silu = lambda X: X * sigmoid(X)
d_silu = lambda X: sigmoid(X) + X * d_sigmoid(X)

leaky_relu = lambda X, slope=0.01: slope * np.minimum(X, 0) + np.maximum(X, 0)
d_leaky_relu = lambda X, slope=0.01: slope * (X <= 0) + (X > 0).astype(float)

identity = lambda X: X
d_identity = lambda X: np.ones(X.shape)

abs_ = lambda X: np.abs(X)
d_abs_ = lambda X: -1 * (X <= 0) + (X > 0)

softplus = lambda X: np.log(1 + np.exp(X))
d_softplus = lambda X: sigmoid(X)

leaky_softplus = lambda X, slope=0.01: slope * X + (1 - slope) * (softplus(X) - np.log(2))
d_leaky_softplus = lambda X, slope=0.01: slope + (1 - slope) * (d_softplus(X))


smooth_abs = lambda X, beta = 10: (2 / beta) * softplus(X) - X - ( (2 / beta) * np.log(2) )
d_smooth_abs = lambda X, beta = 10: (2 / beta) * d_softplus(X) - 1

custom_activations_map = {'ReLU': [relu, d_relu],
               'Leaky-Softplus': [leaky_softplus, d_leaky_softplus], 
               'ELU': [elu, d_elu],
               'GELU': [gelu, d_gelu],
               'SiLU': [silu, d_silu],
               'Leaky-ReLU': [leaky_relu, d_leaky_relu],
               'Abs': [abs_, d_abs_],
               'SmoothAbs': [smooth_abs, d_smooth_abs],
               'Identity': [identity, d_identity]}

# Model Training

In [ ]:
def get_data(data_type, d, K, N_k, nu, whiten_, normalize_):
    # Generate train samples
    if data_type == 'gmm':
        train_set, train_loader = generate_gmm_data(d, K, N_k, nu=nu, whiten_=whiten_, normalize_=normalize_)
        val_set, val_loader = generate_gmm_data(d, K, N_k, nu=nu, whiten_=whiten_, normalize_=normalize_)
    elif data_type == 'uos':
        train_set, train_loader = generate_uos_data(d, K, N_k, subspace_rank=sr, nu=nu, whiten_=whiten_, normalize_=normalize_)
        val_set, val_loader = generate_uos_data(d, K, N_k, subspace_rank=sr, nu=nu, whiten_=whiten_, normalize_=normalize_)
    elif data_type == 'random':
        train_set, train_loader = generate_random_data(d, K, N_k, whiten_=whiten_, normalize_=normalize_)
        val_set, val_loader = generate_random_data(d, K, N_k, whiten_=whiten_, normalize_=normalize_)

    X = train_set.samples.T
    Y = train_set.targets.T
    
    return train_loader, val_loader, X, Y

In [ ]:
def train_model(L, d, h_d, K, act_fn_str, 
                train_loader, 
                val_loader, 
                epochs, 
                lr, 
                freq,
                loss_fn, 
                freeze_last_layer=True, 
                residual=False, 
                batchnorm=False, 
                init_method='orthogonal',
                init_scale=1):
    
    
    # Train model
    model = MLP(L, 
                d, 
                h_d, 
                K, 
                act_fn_str, 
                freeze_last_layer=freeze_last_layer,
                residual=residual, 
                batchnorm=batchnorm, 
                init_method=init_method, 
                init_scale=init_scale)

    model = model.to(device)

    # Training
    criterion = loss_map[loss_fn]
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0, weight_decay=0)
    scheduler = None
    #scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs, eta_min=0)
    results = train(model, 
                    criterion, 
                    optimizer, 
                    train_loader, 
                    val_loader, 
                    epochs, 
                    device,
                    scheduler=scheduler, 
                    return_weights=True,
                    return_grads=True,
                    return_outs=True)


    weights = results['all_weights']
    grads = results['all_grads']
    outs = results['all_outs']
    train_losses = results['train_losses']
    
    return weights, grads, outs, train_losses

# Decomposition of weight matrix during training

In [ ]:
'''
Helper function to calculate the norm of each component in the weight matrix 
'''
def weight_decomp(weights, grads, r, init_scale, epochs):
    
    # Get "small update subspace"
    U_null, V_null = small_update_subspace(weights[0][0], grads[0][0], r, init_scale)
    m = U_null.shape[-1]

    # Get "large update subspace"
    U_update = np.linalg.svd(U_null, full_matrices=True)[0][:, m:]
    V_update = np.linalg.svd(V_null, full_matrices=True)[0][:, m:]
    
    # Concatenate two subspaces to create orthogonal matrices
    U = np.concatenate((U_update, U_null), axis=1)
    V = np.concatenate((V_update, V_null), axis=1)


    norms = np.zeros((epochs, 5))
    for t in range(epochs):
        # Get weights and resulting decomposition
        W_1_t = weights[t][0]
        
        decomp = U.T @ W_1_t @ V
        
        # Calculate norms of each component, and original norm
        total_norm = np.linalg.norm(decomp, ord='fro')**2
        
        norms[t, 0] = total_norm
        norms[t, 1] = np.linalg.norm(decomp[:2*K, :2*K], ord='fro')**2 / total_norm
        norms[t, 2] = np.linalg.norm(decomp[:2*K, 2*K:], ord='fro')**2 / total_norm
        norms[t, 3] = np.linalg.norm(decomp[2*K:, :2*K], ord='fro')**2 / total_norm
        norms[t, 4] = np.linalg.norm(decomp[2*K:, 2*K:], ord='fro')**2 / total_norm
        
        
    return norms

In [ ]:
reduction = 'sum'
loss_map = {
    'ce_loss': nn.CrossEntropyLoss(reduction=reduction),
    'mse_loss': nn.MSELoss(reduction=reduction)
}

data_type_map = {
    'gmm': 'GMM', 
    'uos': 'UoS',
    'random': 'Random'
}

In [ ]:
# Data setup
d = 32 # Ambient dim
h_d = d
p = d # Intrinsic dim 
K = 4 # Number of classes
N_k = 250
N = N_k * K
nu = 1

r = np.minimum(K, p)
sr = 2*r

# Model setup
L = 2
act_fn_strs = ['Leaky-Softplus', 'Leaky-ReLU', 'GELU', 'ReLU']
#act, d_act = custom_activations_map[act_fn_str]
residual = False
batchnorm = False
freeze_last_layer = True
init_method = 'orthogonal'
init_scale = 0.01

whiten_ = True
normalize_ = False

# Training setup
epochs = 50
lr = 1e-2
freq = 1
#loss_fn = 'ce_loss'
loss_fn = 'mse_loss'

data_type = 'gmm'


In [ ]:
trials = 10

weight_decomp_norms = np.zeros((trials, len(act_fn_strs), epochs, 5))
for (k, act_fn_str) in enumerate(act_fn_strs):
    print("===== ACTIVATION FUNCTION: ", act_fn_str, " =====")
    for i in range(trials):
        print("=== TRIAL " + str(i) + " ===")
        # Train model
        train_loader, val_loader, _, _ = get_data(data_type, d, K, N_k, nu, whiten_, normalize_)
        weights, grads, outs, train_losses = train_model(L, d, h_d, K, act_fn_str, 
                                                         train_loader, 
                                                         val_loader, 
                                                         epochs, 
                                                         lr, 
                                                         freq, 
                                                         loss_fn, 
                                                         freeze_last_layer=freeze_last_layer, 
                                                         residual=residual,
                                                         batchnorm=batchnorm,
                                                         init_method=init_method,
                                                         init_scale=init_scale)


        # Plot train loss
        # plot_train_losses(train_losses, act_fn_str, d, p, K, None, savefig=False)



        # Calculate components of decomposition
        weight_decomp_norms[i, k, ...] = weight_decomp(weights, grads, r, init_scale, epochs)




In [ ]:
# Mean and stdev of weight decomposition norms across trials
mean_weight_decomp_norms = np.mean(weight_decomp_norms, axis=0)
std_weight_decomp_norms = np.std(weight_decomp_norms, axis=0) # / np.sqrt(trials)

# Plotting
x_range = np.arange(mean_weight_decomp_norms.shape[1] - 1) + 1

for (k, act_fn_str) in enumerate(act_fn_strs):
    plt.figure(figsize=(8, 6))
    for j in range(1, mean_weight_decomp_norms[k, ...].shape[-1]):
        plt_label = r'$\widetilde{W}_1^{(' + str(j) + r')}(t)$' if j > 0 else r'$W_1(t)$'
        plt.plot(x_range, mean_weight_decomp_norms[k, 1:, j], label=plt_label)
        plt.fill_between(x_range, 
                         mean_weight_decomp_norms[k, 1:, j] - std_weight_decomp_norms[k, 1:, j], 
                         mean_weight_decomp_norms[k, 1:, j] + std_weight_decomp_norms[k, 1:, j],
                         alpha=0.5)


    #plt.yscale('log')
    plt.title(r'Block Matrix Norm Proportions in $W_1(t)$'
              '\n'
              r'(' + act_fn_str + r', $\epsilon = $' + str(init_scale) + r', $\eta = $' + str(lr) + ')', fontsize=18)
    plt.xlabel('Training Epoch', fontsize=15)
    plt.ylabel('Squared Frobenius Norms', fontsize=15)
    plt.xticks(size=12)
    plt.yticks(size=12)
    
    plt.grid()
    plt.legend(ncol=2, fontsize=12)
    
    savefig_dir = './figs/theory_fixed_W2_decomp_W1/'
    if not os.path.exists(savefig_dir):
        os.makedirs(savefig_dir)
    
    plt.savefig(savefig_dir + act_fn_str + '_W1_block_norms_eps_' + str(init_scale) + '_eta_' + str(lr) + '.pdf')
    plt.show() 
    plt.close()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

dig = 2
labels = [
    r'$\widetilde{W}_1^{(1)}(t)$ (' + str(round(mean_weight_decomp_norms[1, 1, 1] * 100, dig)) + '%)',
    r'$\widetilde{W}_1^{(2)}(t)$ (' + str(round(mean_weight_decomp_norms[1, 1, 2] * 100, dig)) + '%)',
    r'$\widetilde{W}_1^{(3)}(t)$ (' + str(round(mean_weight_decomp_norms[1, 1, 3] * 100, dig)) + '%)',
    r'$\widetilde{W}_1^{(4)}(t)$ (' + str(round(mean_weight_decomp_norms[1, 1, 4] * 100, dig)) + '%)'
]

percs = [mean_weight_decomp_norms[1, 1, i] for i in range(1, 5)]

fig, ax = plt.subplots()

wedges, _ = ax.pie(
    percs,
    labels=None,
    wedgeprops={'linewidth': 1, 'edgecolor': 'white'},
    startangle=90
)

kw = dict(
    arrowprops=dict(arrowstyle="-"),
    va="center",
    fontsize=14,
)

for idx, (w, label) in enumerate(zip(wedges, labels)):
    ang = (w.theta2 + w.theta1) / 2
    base_x = np.cos(np.deg2rad(ang))
    base_y = np.sin(np.deg2rad(ang))

    # Default side based on angle
    ha = "left" if base_x > 0 else "right"
    text_x = base_x
    text_y = base_y

    # -------------------------------------------------
    # Force slices 3 and 4 to opposite sides / spacing
    # -------------------------------------------------
    if idx == 2:      # Slice 3 → LEFT side
        ha = "right"
        text_x = 2
        text_y = base_y + 0.15  # slight upward shift

    elif idx == 3:    # Slice 4 → RIGHT side
        ha = "left"
        text_x = 1
        text_y = base_y - 0.15  # slight downward shift
    # -------------------------------------------------

    # Place labels completely OUTSIDE the pie (radius ~1.35)
    ax.annotate(
        label,
        xy=(0.75 * base_x, 0.75 * base_y),     # connection point on wedge
        xytext=(1.35 * text_x, 1.35 * text_y), # LABEL OUTSIDE PIE
        horizontalalignment=ha,
        **kw
    )

ax.set_aspect("equal")
plt.tight_layout()
plt.title(r'Squared Norm Proportions at $t = 1$' 
          '\n' 
          r'(Leaky ReLU, $\epsilon = ' + str(init_scale) + r'$, $\eta = ' + str(lr) + r'$)', fontsize=15)
# plt.savefig(savefig_dir + 'Leaky_ReLU_W1_block_norm_props_t1_eps_' + str(init_scale) + '_eta_' + str(lr) + '.pdf', bbox_inches='tight')
plt.show()
plt.close()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

dig = 2
labels = [
    r'$\widetilde{W}_1^{(1)}(t)$ (' + str(round(mean_weight_decomp_norms[3, 1, 1] * 100, dig)) + '%)',
    r'$\widetilde{W}_1^{(2)}(t)$ (' + str(round(mean_weight_decomp_norms[3, 1, 2] * 100, dig)) + '%)',
    r'$\widetilde{W}_1^{(3)}(t)$ (' + str(round(mean_weight_decomp_norms[3, 1, 3] * 100, dig)) + '%)',
    r'$\widetilde{W}_1^{(4)}(t)$ (' + str(round(mean_weight_decomp_norms[3, 1, 4] * 100, dig)) + '%)'
]

percs = [mean_weight_decomp_norms[3, 1, i] for i in range(1, 5)]

fig, ax = plt.subplots()

wedges, _ = ax.pie(
    percs,
    labels=None,
    wedgeprops={'linewidth': 1, 'edgecolor': 'white'},
    startangle=90
)

kw = dict(
    arrowprops=dict(arrowstyle="-"),
    va="center",
    fontsize=14,
)

for idx, (w, label) in enumerate(zip(wedges, labels)):
    ang = (w.theta2 + w.theta1) / 2
    base_x = np.cos(np.deg2rad(ang))
    base_y = np.sin(np.deg2rad(ang))

    # Default side based on angle
    ha = "left" if base_x > 0 else "right"
    text_x = base_x
    text_y = base_y

    # -------------------------------------------------
    # Force slices 2 - 4 to opposite sides / spacing
    # -------------------------------------------------
    if idx == 2:      # Slice 3 → LEFT side
        ha = "right"
        text_x = 2
        text_y = base_y + 0.15  # slight upward shift

    elif idx == 3:    # Slice 4 → RIGHT side
        ha = "left"
        text_x = 1
        text_y = base_y - 0.15  # slight downward shift
    # -------------------------------------------------

    # Place labels completely OUTSIDE the pie (radius ~1.35)
    ax.annotate(
        label,
        xy=(0.75 * base_x, 0.75 * base_y),     # connection point on wedge
        xytext=(1.35 * text_x, 1.35 * text_y), # LABEL OUTSIDE PIE
        horizontalalignment=ha,
        **kw
    )

ax.set_aspect("equal")
plt.tight_layout()
plt.title(r'Squared Norm Proportions at $t = 1$' 
          '\n' 
          r'(ReLU, $\epsilon = ' + str(init_scale) + r'$, $\eta = ' + str(lr) + r'$)', fontsize=15)
plt.savefig(savefig_dir + 'ReLU_W1_block_norm_props_t1_eps_' + str(init_scale) + '_eta_' + str(lr) + '.pdf', bbox_inches='tight')
plt.show()
plt.close()


# Gradient components

In [ ]:
'''
Helper function to determine if Delta_1(t) lies completely in null(X)
'''
def gradient_null_X(weights, outs, act_fn_str, X, Y, epochs):
    d, N = X.shape
    
    null_X_perp = np.linalg.svd(X).Vh[:d, :].T
    
    act_fn, d_act_fn = custom_activations_map[act_fn_str]
    W2 = weights[0][1]
    
    proj_norms = np.zeros(epochs)
    for t in range(epochs):
        W1_t = weights[t][0]
        Delta2_t = outs[t].T - Y
        Delta1_t = (W2.T @ Delta2_t) * d_act_fn(W1_t @ X)
        
        proj_null_X_perp = null_X_perp @ (null_X_perp.T @ Delta1_t.T)
        
        proj_norms[t] = np.linalg.norm(proj_null_X_perp, ord='fro')**2 #/ np.linalg.norm(Delta1_t, ord='fro')**2
        
    return proj_norms
    

In [ ]:
reduction = 'sum'
loss_map = {
    'ce_loss': nn.CrossEntropyLoss(reduction=reduction),
    'mse_loss': nn.MSELoss(reduction=reduction)
}

data_type_map = {
    'gmm': 'GMM', 
    'uos': 'UoS',
    'random': 'Random'
}

In [ ]:
# Data setup
d = 32 # Ambient dim
h_d = d
p = d # Intrinsic dim 
K = 4 # Number of classes
N_k = 250
N = N_k * K
nu = 1

r = np.minimum(K, p)

# Model setup
L = 2
act_fn_strs = ['Leaky-Softplus', 'GELU'] #'Leaky-ReLU', 'GELU', 'ReLU']
#act, d_act = custom_activations_map[act_fn_str]
residual = False
batchnorm = False
freeze_last_layer = True
init_method = 'orthogonal'
init_scale = 0.1

whiten_ = True
normalize_ = False

# Training setup
epochs = 500
lr = 1e-2
freq = epochs + 1
#loss_fn = 'ce_loss'
loss_fn = 'mse_loss'

data_type = 'gmm'


In [ ]:
trials = 10

proj_norms = np.zeros((trials, len(act_fn_strs), epochs))
for (k, act_fn_str) in enumerate(act_fn_strs):
    print("===== ACTIVATION FUNCTION: ", act_fn_str, " =====")
    for i in range(trials):
        print("=== TRIAL " + str(i) + " ===")
        # Train model
        train_loader, val_loader, X, Y = get_data(data_type, d, K, N_k, nu, whiten_, normalize_)
        weights, grads, outs, train_losses = train_model(L, d, h_d, K, act_fn_str, 
                                                         train_loader, 
                                                         val_loader, 
                                                         epochs, 
                                                         lr, 
                                                         freq, 
                                                         loss_fn, 
                                                         freeze_last_layer=freeze_last_layer, 
                                                         residual=residual,
                                                         batchnorm=batchnorm,
                                                         init_method=init_method,
                                                         init_scale=init_scale)
        
        proj_norms[i, k, :] = gradient_null_X(weights, outs, act_fn_str, X, Y, epochs)





In [ ]:
# Mean and stdev of weight decomposition norms across trials
mean_proj_norms = np.mean(proj_norms, axis=0)
std_proj_norms = np.std(proj_norms, axis=0) #/ np.sqrt(trials)

# Plotting
start = 200
x_range = np.arange(start, mean_proj_norms.shape[1])

plt.figure(figsize=(8, 6))
for (k, act_fn_str) in enumerate(act_fn_strs):    
    plt.plot(x_range, mean_proj_norms[k, start:], label=act_fn_str)
    plt.fill_between(x_range, 
                     mean_proj_norms[k, start:] - std_proj_norms[k, start:], 
                     mean_proj_norms[k, start:] + std_proj_norms[k, start:],
                     alpha=0.5)


    
plt.title(r'Norm of $\Delta_1(t)$ in $\mathcal{N}^\perp(X)$'
          '\n'
          r'($\epsilon = $' + str(init_scale) + r', $\eta = $' + str(lr) + ')', fontsize=18)
plt.xlabel('Training Epoch', fontsize=15)
plt.ylabel('Squared Frobenius Norm', fontsize=15)
plt.xticks(size=12)
plt.yticks(size=12)
if start > 0:
    plt.ylim(bottom=-0.1)

plt.grid()
plt.legend(fontsize=12)

savefig_dir = './figs/theory_fixed_W2_decomp_W1/'
if not os.path.exists(savefig_dir):
    os.makedirs(savefig_dir)

plt.savefig(savefig_dir + 'smooth_acts_Delta1_t_null_X_perp_eps_' + str(init_scale) + '_eta_' + str(lr) + '_start_' + str(start) + '.pdf')
plt.show() 
plt.close()

# Weight update subspaces

In [ ]:
savefig = False
net_dir = act_fn_str + '_network_L_' + str(L) + '_' + init_method + '_init_scale_' + str(init_scale) 
data_dir = data_type + '_synthetic_data_d_' + str(d) + '_K_' + str(K) + '_N_' + str(N)
if whiten_:
    data_dir = 'whitened_' + data_dir
elif normalize_ and not whiten_:
    save_path = 'normalized_' + data_dir
else:
    save_path = 'raw_' + data_dir

save_path = os.path.join('./figs', data_dir, net_dir)
if savefig and not os.path.exists(save_path):
    os.makedirs(save_path)
    
subspace_type = 1
result = save_weight_svd_errors(weights, grads, E_0, X, r, init_scale, subspace_type)

plot_weight_svd_errors(result, act_fn_str, d, p, K, save_path, subspace_type, data_type.capitalize(), savefig)
    

# all_rs = np.arange(1, 2*r + 1)

# for r_ in all_rs:
#     print("K = ", K)
#     print("r =", r_)
#     result = save_weight_svd_errors(weights, grads, E_0, X, r_, init_scale, subspace_type)

#     plot_weight_svd_errors(result, act_fn_str, d, p, K, save_path, subspace_type, data_type.capitalize(), savefig)
    
#     print("\n")

In [ ]:
print(result['svals'][0][0])
print(result['svals'][-1][0])

In [ ]:
# result = save_weight_svd_angles(weights, grads, r, init_scale)
# plot_weight_svd_angles(result, act_fn, d, p, K, save_path, savefig)

# Gradient subspaces

In [ ]:
W_1_init = weights[0][0]
W_2_init = weights[0][1]

Gamma_0 = W_2_init.T @ E_0 @ X.T

U_Gam, _, V_Gam = np.linalg.svd(Gamma_0)
#U_Gam = W_1_init.T @ U_Gam / init_scale
V_Gam = V_Gam.T

G_1_init = grads[0][0]

U_grad, _, V_grad = np.linalg.svd(G_1_init)
V_grad = V_grad.T


print(subspace_error(U_Gam[:, K:], U_grad[:, K:]))
print(subspace_error(V_Gam[:, K:], V_grad[:, K:]), "\n")


U_errors = []
V_errors = []
d_M_changes = []
for t in range(1, epochs):

    W_cur = weights[t][0]
    W_prev = weights[t - 1][0]
    
    G_cur = grads[t][0]
    G_prev = grads[t - 1][0]
    
    E_t = outs[t].T - Y
    
    
    U_grad_cur, _, V_grad_cur = np.linalg.svd(G_cur)
    V_grad_cur = V_grad_cur.T
    
    U_grad_prev, _, V_grad_prev = np.linalg.svd(G_prev)
    V_grad_prev = V_grad_prev.T
    
    
    U_errors.append(subspace_error(U_grad_cur[:, :K], U_grad_prev[:, :K]))
    V_errors.append(subspace_error(V_grad_cur[:, :K], V_grad_prev[:, :K]))
    
    
    M_cur = np.int8( (W_cur @ X) > 0 )
    M_prev = np.int8( (W_prev @ X) > 0 )
    
    d_M = M_cur - M_prev
    d_M_changes.append(np.sum(d_M != 0) / (d * N))


plt.figure()
plt.title(r'$G_1(t)$ Sing. Sub. Distances (' + act_fn_str + r', L = 2, $\epsilon = $' + str(init_scale) + r', $\eta = $' + str(lr) + ')')
plt.plot(np.arange(epochs - 1) + 1, U_errors, label='Left singular subspace')
plt.plot(np.arange(epochs - 1) + 1, V_errors, label='Right singular subspace')
plt.grid('on')
plt.legend()
if savefig:
    plt.savefig('./figs/L_2_fixed_W2_theory_verification/' + act_fn_str + '_G1_sing_sub_dist_fixed_W2_L_2_eps_' + str(init_scale) + '_eta_' + str(lr) + '.pdf')
plt.show()
plt.close()

plt.figure()
plt.title(r'Proportion of sign changes in $\Delta M(t)$ (L = 2, $\epsilon = $' + str(init_scale) + r', $\eta = $' + str(lr) + ')')
plt.plot(np.arange(epochs - 1) + 1, d_M_changes)
plt.grid('on')
if savefig:
    plt.savefig('./figs/L_2_fixed_W2_theory_verification/' + act_fn_str + '_W1_X_mask_diffs_fixed_W2_L_2_eps_' + str(init_scale) + '_eta_' + str(lr) + '.pdf')
plt.show()
plt.close()
    


# Weight matrix decomposition

In [ ]:
U_null, V_null = small_update_subspace(weights[0][0], grads[0][0], r, init_scale)
m = U_null.shape[-1]


U_update = np.linalg.svd(U_null, full_matrices=True)[0][:, m:]
V_update = np.linalg.svd(V_null, full_matrices=True)[0][:, m:]

U = np.concatenate((U_update, U_null), axis=1)
V = np.concatenate((V_update, V_null), axis=1)


W2_s1 = np.linalg.norm(weights[0][1], ord=2)
loss_init = np.linalg.norm(E_0, ord='fro')
gamma = np.exp(1) / np.sqrt(2 * np.pi)
mu = 0.8


beta = gamma**2 * W2_s1**2 + mu * W2_s1 * loss_init

print("beta = ", beta)


rel_norms = np.zeros((5, epochs))
for t in range(epochs):
    W_1_t = weights[t][0]
    G_1_t = grads[t][0]
    
#     print(lr)
#     print(lr**2)
    
#     print(np.linalg.norm(G_1_t @ V_null, ord='fro'))
#     print(np.linalg.norm(W_1_t @ V_null - init_scale * U_null, ord='fro'))
    
#     print(np.linalg.norm(W_1_t, ord=2), "\n")
    
    decomp = U.T @ W_1_t @ V
    
    total_norm = np.linalg.norm(decomp, ord='fro')**2
    
    rel_norms[0, t] = np.linalg.norm(decomp[:2*K, :2*K], ord='fro')**2 #/ total_norm
    rel_norms[1, t] = np.linalg.norm(decomp[:2*K, 2*K:], ord='fro')**2 #/ total_norm
    rel_norms[2, t] = np.linalg.norm(decomp[2*K:, :2*K], ord='fro')**2 #/ total_norm
    rel_norms[3, t] = np.linalg.norm(decomp[2*K:, 2*K:], ord='fro')**2 #/ total_norm
    rel_norms[4, t] = total_norm



plt.figure()
# plt.semilogy(np.arange(rel_norms.shape[1]), rel_norms[0, :], label=r'$\widetilde{W}_1^{(1)}(t)$')
# plt.semilogy(np.arange(rel_norms.shape[1]), rel_norms[1, :], label=r'$\widetilde{W}_1^{(2)}(t)$')
# plt.semilogy(np.arange(rel_norms.shape[1]), rel_norms[2, :], label=r'$\widetilde{W}_1^{(3)}(t)$')
# plt.semilogy(np.arange(rel_norms.shape[1]), rel_norms[3, :], label=r'$\widetilde{W}_1^{(4)}(t)$')
# plt.semilogy(np.arange(rel_norms.shape[1]), rel_norms[4, :], label=r'$W_1(t)$')
plt.plot(np.arange(rel_norms.shape[1]), rel_norms[0, :], label=r'$\widetilde{W}_1^{(1)}(t)$')
plt.plot(np.arange(rel_norms.shape[1]), rel_norms[1, :], label=r'$\widetilde{W}_1^{(2)}(t)$')
plt.plot(np.arange(rel_norms.shape[1]), rel_norms[2, :], label=r'$\widetilde{W}_1^{(3)}(t)$')
plt.plot(np.arange(rel_norms.shape[1]), rel_norms[3, :], label=r'$\widetilde{W}_1^{(4)}(t)$')
plt.plot(np.arange(rel_norms.shape[1]), rel_norms[4, :], label=r'$W_1(t)$')
plt.title(r'Block Matrix Norms in $W_1(t)$ (' + act_fn_str + r', L = 2, $\epsilon = $' + str(init_scale) + r', $\eta = $' + str(lr) + ')')
plt.ylabel('Squared Frobenius Norms')
plt.xlabel('Epoch')
plt.grid()
plt.legend()
# plt.savefig('./figs/L_2_fixed_W2_theory_verification/' + act_fn_str + '_W1_block_norms_eps_' + str(init_scale) + '_eta_' + str(lr) + '.pdf')
plt.show() 
plt.close()
    

# Pre-activations

In [ ]:
W2_svals = np.linalg.svd(weights[0][1], compute_uv=False)
W2_smin = W2_svals[-1]
W2_smax = W2_svals[0]

loss_init = np.linalg.norm(E_0, ord='fro')**2
gamma = np.exp(1) / np.sqrt(2 * np.pi)
mu = 2 / np.sqrt(2*np.pi)

rho = 0.5 * W2_smin / (np.sqrt(mu) * W2_smax)
print("rho = ", rho)


beta = gamma**2 * W2_s1**2 + mu * W2_s1 * loss_init
for t in range(epochs + 1):
    print("===== t =", t, " =====")
    H_prev = X
    for l in range(L - 1):
        print("=== l =", l + 1, " ===")
        
        W_l_t = weights[t][l]
        print("||W_l(t)||_F = ", np.linalg.norm(W_l_t, ord='fro'))
        
        Z_l_t = W_l_t @ H_prev  


        print("W_l(t) max = ", np.max(np.abs(W_l_t)))
        print("Z_l(t) max = ", np.max(np.abs(Z_l_t)))
        
        print("W_l(t) mean = ", np.mean(np.abs(W_l_t)))
        print("Z_l(t) mean = ", np.mean(np.abs(Z_l_t)))
        
        
        print("W_l(t) variance = ", np.std(np.abs(W_l_t)**2))
        print("Z_l(t) variance = ", np.std(np.abs(Z_l_t)**2))

        print("\n")
        
        H_prev = act(Z_l_t)
        
    print("\n")
    
    

In [ ]:
W2 = weights[0][1]
W2_svals = np.linalg.svd(W2, compute_uv=False)
W2_smin = W2_svals[-1]
W2_smax = W2_svals[0]

alpha = 0.5
loss_init = np.linalg.norm(E_0, ord='fro')**2
gamma = np.exp(1) / np.sqrt(2 * np.pi)
mu = 2 / np.sqrt(2*np.pi)

C_1 = W2_smax * (mu / 2)
C_2 = W2_smax * mu + W2_smax**2 * alpha

R1 = W2_smin / (4 * C_2)
R2 = W2_smin / (8 * C_1)

print(R1)
print(R2)


beta = gamma**2 * W2_s1**2 + mu * W2_s1 * loss_init
for t in range(epochs):
    W1_t = weights[t][0]
    lin_out = ((W2 @ W1_t) @ X) / 2
    
    G1_t_lin = W2.T @ (lin_out - Y) @ X.T
    G1_t = grads[t][0]
    
    #print("Lin out = ", lin_out) #np.linalg.norm(outs[t].T - lin_out, ord='fro'))
    
    print("W1(t) = ", np.linalg.norm(W1_t))
    print("G1(t) = ", np.linalg.norm(G1_t))
    print("G1(t) lin = ", np.linalg.norm(G1_t_lin))
    print("grad diff = ", np.linalg.norm(G1_t - G1_t_lin), "\n")
    
    
    
    

# Gradient singular values

In [ ]:
idx = -1
grads_idx = grads[idx]

for (l, G_l) in enumerate(grads_idx[:-1]):
    s = svd_(G_l, compute_uv=False)
    plt.figure()
    plt.scatter(np.arange(len(s)) + 1, s)
    plt.title(r'$G_' + str(l + 1) + r'(' + str(idx) + r')$ Singular Values (' + act_fn_str + ', ' + data_type + ' Whitened Data)', fontsize=12 )
    plt.xlabel('SV Index', fontsize=12)
    plt.ylabel('Singular Value', fontsize=12)
    plt.show()
    plt.close()



# Gradient - subspace basis products

In [ ]:
W1_0 = weights[0][0]
W2_0 = weights[0][1]
G1_0 = grads[0][0]

E_0 = (W2_0 @ act(W1_0 @ X) - Y)

U_null, V_null = small_update_subspace(W1_0, G1_0, K, init_scale)
U_null2, V_null2 = small_update_subspace_2(weights[0], E_0, X, K, init_scale)

print(subspace_error(V_null, V_null2))
print(subspace_error(U_null, U_null2))


X_V_prod_norms = []
G_V_prod_norms = []

W1_V_prod_norms = []
D_W1_V_prod_norms = []
G_W1_V_prod_norms = []

G_norms = []
G_W1_norms = []

D_U_prod_norms = []
G_U_prod_norms = []

W2_E_X_norms = []
W2_E_X_V_norms = []


for t in range(epochs):
    G1_t = grads[t][0]
    W1_t = weights[t][0]
    W2_t = weights[t][1]
    Z2_t = W2_t @ act(W1_t @ X)
    E_t = Z2_t - Y
    

    Delta1_1_t = W2_t.T @ E_t
    Delta1_2_t = d_act(W1_t @ X)

    Delta_1_t = Delta1_1_t * Delta1_2_t
    G1_t_an = Delta_1_t @ X.T
    
    
    G1_t_norm = np.linalg.norm(G1_t_an, ord='fro')
    G_W1_t_norm = np.linalg.norm(G1_t_an.T @ W1_t, ord='fro')
    
    G_norms.append(G1_t_norm)
    G_W1_norms.append(G_W1_t_norm)
    
    X_V_prod_norms.append(np.linalg.norm(X.T @ V_null2, ord='fro'))
    G_V_prod_norms.append(np.linalg.norm(G1_t_an @ V_null2, ord='fro'))
    
    W1_V_prod_norms.append(np.linalg.norm(W1_t @ V_null2, ord='fro'))
    D_W1_V_prod_norms.append(np.linalg.norm(Delta_1_t.T @ W1_t @ V_null2, ord='fro'))
    G_W1_V_prod_norms.append(np.linalg.norm(G1_t_an.T @  W1_t @ V_null2))
    
    D_U_prod_norms.append(np.linalg.norm(Delta_1_t.T @ U_null2, ord='fro'))
    G_U_prod_norms.append(np.linalg.norm(G1_t_an.T @ U_null2, ord='fro'))
    
    
    A_t = W2_t.T @ E_t @ X.T
    W2_E_X_norms.append(np.linalg.norm(A_t, ord='fro'))
    W2_E_X_V_norms.append(np.linalg.norm(A_t @ V_null2, ord='fro'))
    
    


plt.figure()
plt.plot(np.arange(epochs) + 1, G_norms, label=r'$G_1(t)$')
plt.plot(np.arange(epochs) + 1, X_V_prod_norms, label=r'$X^\top V$')
plt.plot(np.arange(epochs) + 1, G_V_prod_norms, label=r'$\Delta_1(t) X^\top V$')
plt.title('V Matrix Product Frobenius Norms (' + act_fn_str + ')')
plt.legend()
plt.grid()
plt.show()
plt.close()

plt.figure()
plt.plot(np.arange(epochs) + 1, G_norms, label=r'$G_1(t)$')
plt.plot(np.arange(epochs) + 1, D_U_prod_norms, label=r'$\Delta_1^\top(t) U$')
plt.plot(np.arange(epochs) + 1, G_U_prod_norms, label=r'$X \Delta_1^\top(t) U$')
plt.title('U Matrix Product Frobenius Norms (' + act_fn_str + ')')
plt.legend()
plt.grid()
plt.show()
plt.close()

plt.figure()
plt.plot(np.arange(epochs) + 1, W2_E_X_norms, label=r'$W_2(t)^\top E(t) X$')
plt.plot(np.arange(epochs) + 1, W2_E_X_V_norms, label=r'$W_2(t)^\top E(t) X^\top V$')
plt.legend()
plt.grid()
plt.show()
plt.close()

print(W2_E_X_V_norms[0])


# Perturbation matrix

In [ ]:
phi_neg = lambda x: np.exp(x) - 1

W1_0 = weights_init[0]
W2_0 = weights_init[1]
G1_0 = grads[0][0]

M_bar = (W1_0 @ X) <= 0

Delta_2_0 = (W2_0 @ elu(W1_0 @ X) - Y)
W2_D2 = W2_0.T @ Delta_2_0
W2_Gamma = W2_D2 @ X.T

P0_no_mask = W2_D2 * phi_neg(W1_0 @ X) 
P0 = P0_no_mask * M_bar


P0_tilde_no_mask = P0_no_mask @ X.T
P0_tilde = P0 @ X.T


#U_null, V_null = small_update_subspace(W1_0, G1_0, K, init_scale)
U_null, V_null = small_update_subspace_2(weights_init, Delta_2_0, X, K, init_scale)

S1 = svd_(G1_0)[-1][:, K:]
S2 = svd_(G1_0.T @ W1_0)[-1][:, K:]


S1_tilde = svd_(W2_Gamma)[-1][:, K:]
S2_tilde = svd_(W2_Gamma.T @ W1_0)[-1][:, K:]


# print(subspace_error(S1, S1_tilde))
# print(subspace_error(S2, S2_tilde))
# print(subspace_error(V_null, V_null2))
# print(np.linalg.norm(G1_0 @ V_null,ord='fro'))
# print(np.linalg.norm(G1_0 @ V_null2,ord='fro'))


P0_tilde_V = P0_tilde @ V_null
P0_tilde_no_mask_V = P0_tilde_no_mask @ V_null
P0_tilde_U = P0_tilde.T @ U_null
P0_tilde_no_mask_U = P0_tilde_no_mask.T @ U_null


phi_neg_norms = [np.linalg.norm(phi_neg(W1_0 @ X), ord='fro')]
G1_norms = [np.linalg.norm(G1_0, ord='fro')]
W2_D2_norms = [np.linalg.norm(W2_D2, ord='fro')]
W2_Gamma_norms = [np.linalg.norm(W2_Gamma, ord='fro')]
W2_Gamma_V_norms = [np.linalg.norm(W2_Gamma @ V_null, ord='fro')]
W2_Gamma_U_norms = [np.linalg.norm(W2_Gamma.T @ U_null, ord='fro')]

P_norms = [np.linalg.norm(P0, ord='fro')]
P_no_mask_norms = [np.linalg.norm(P0_no_mask, ord='fro')]

P_tilde_norms = [np.linalg.norm(P0_tilde, ord='fro')]
P_tilde_no_mask_norms = [np.linalg.norm(P0_tilde_no_mask, ord='fro')]

P_tilde_V_norms = [np.linalg.norm(P0_tilde_V, ord='fro')]
P_tilde_no_mask_V_norms = [np.linalg.norm(P0_tilde_no_mask_V, ord='fro')]

P_tilde_U_norms = [np.linalg.norm(P0_tilde_U, ord='fro')]
P_tilde_no_mask_U_norms = [np.linalg.norm(P0_tilde_no_mask_U, ord='fro')]


for t in range(1, epochs):
    print('===== EPOCH ' + str(t) + ' =====')
    G1_t = grads[t][0]
    W1_t = weights[t][0]
    W2_t = weights[t][1]
    
    G1_norms.append(np.linalg.norm(G1_t, ord='fro'))
    phi_neg_norms.append(np.linalg.norm(phi_neg(W1_t @ X), ord='fro'))
    
    Delta_2_t = (W2_t @ elu(W1_t @ X) - Y)
    W2_D2 = W2_t.T @ Delta_2_t
    W2_D2_norms.append(np.linalg.norm(W2_D2, ord='fro'))
    
    W2_Gamma = W2_D2 @ X.T
    W2_Gamma_norms.append(np.linalg.norm(W2_Gamma, ord='fro'))
    W2_Gamma_V_norms.append(np.linalg.norm(W2_Gamma @ V_null, ord='fro'))
    W2_Gamma_U_norms.append(np.linalg.norm(W2_Gamma.T @ U_null, ord='fro'))
    
    
    Pt_no_mask = W2_D2 * phi_neg(W1_t @ X)
    Pt = Pt_no_mask * M_bar
    
    P_norms.append(np.linalg.norm(Pt, ord='fro'))
    P_no_mask_norms.append(np.linalg.norm(Pt_no_mask, ord='fro'))
    
    Pt_tilde = Pt @ X.T
    Pt_tilde_no_mask = Pt_no_mask @ X.T
    P_tilde_norms.append(np.linalg.norm(Pt_tilde, ord='fro'))
    P_tilde_no_mask_norms.append(np.linalg.norm(Pt_tilde_no_mask, ord='fro'))
    
    
    Pt_tilde_V = Pt_tilde @ V_null
    Pt_tilde_U = Pt_tilde.T @ U_null
    P_tilde_V_norms.append(np.linalg.norm(Pt_tilde_V, ord='fro'))
    P_tilde_U_norms.append(np.linalg.norm(Pt_tilde_U, ord='fro'))
    
    Pt_tilde_no_mask_V = Pt_tilde_no_mask @ V_null
    Pt_tilde_no_mask_U = Pt_tilde_no_mask.T @ U_null
    P_tilde_no_mask_V_norms.append(np.linalg.norm(Pt_tilde_no_mask_V, ord='fro'))
    P_tilde_no_mask_U_norms.append(np.linalg.norm(Pt_tilde_no_mask_U, ord='fro'))
    
    
    #Ut_null, Vt_null = small_update_subspace_2(weights[t], Delta_2_t, X, K, init_scale)
    
#     print(np.linalg.norm(W2_Gamma @ V_null, ord='fro'))
#     print(np.linalg.norm(W2_Gamma, ord='fro'))

    print("W_2(t).T Gamma(0) svals = ", np.linalg.svd(W2_Gamma, compute_uv=False))
    print("P(t) X.T svals = ", np.linalg.svd(Pt_tilde, compute_uv=False))
    print("W_2(t)^.T Gamma(t) + P(t)X.T svals = ", np.linalg.svd(W2_Gamma + Pt_tilde, compute_uv=False))


    
#     Pt_tilde_svals = np.linalg.svd(Pt_tilde, compute_uv=False)
#     Pt_tilde_no_mask_svals = np.linalg.svd(Pt_tilde_no_mask, compute_uv=False)
#     W2_Gamma_svals = np.linalg.svd(W2_Gamma, compute_uv=False)

#     plt.figure()
#     plt.scatter(np.arange(len(Pt_tilde_svals)), Pt_tilde_svals, label=r'$P(t)$ with mask')
#     plt.scatter(np.arange(len(Pt_tilde_no_mask_svals)), Pt_tilde_no_mask_svals, label=r'$P(t)$ without mask')
#     plt.scatter(np.arange(len(W2_Gamma_svals)), W2_Gamma_svals, label=r'$W_2^\top(t) \Gamma(t)$')
#     plt.legend()
#     plt.title('Epoch ' + str(t))
#     #plt.yscale('log')
#     plt.show()
#     plt.close()
    
    

plt.figure()
plt.plot(np.arange(epochs), P_norms, label=r'$\|P(t)\|_F$ with mask')
plt.plot(np.arange(epochs), P_tilde_norms, label=r'$\|P(t) X^\top\|_F$ with mask')
plt.plot(np.arange(epochs), P_tilde_V_norms, label=r'$\|P(t) X^\top V\|_F$ with mask')
plt.plot(np.arange(epochs), P_tilde_U_norms, label=r'$\|X P^\top(t) U\|_F$ with mask')

plt.plot(np.arange(epochs), P_no_mask_norms, label=r'$\|P(t)\|_F$ without mask')
plt.plot(np.arange(epochs), P_tilde_no_mask_norms, label=r'$\|P(t) X^\top\|_F$ without mask')
plt.plot(np.arange(epochs), P_tilde_no_mask_V_norms, label=r'$\|P(t) X^\top V\|_F$ without mask')
plt.plot(np.arange(epochs), P_tilde_no_mask_U_norms, label=r'$\|X P^\top(t) U\|_F$ without mask')
#plt.plot(np.arange(epochs), G1_norms, label=r'$\|G_1(t)\|_F$')
#plt.plot(np.arange(epochs), W2_Gamma_norms, label=r'$\|W_2(t) \Delta_2(t)\|_F$')
plt.legend(ncol=2)
plt.title('Permutation Matrix Product Norms')
plt.grid('on')
plt.ylim([0,6])
plt.show()
plt.close()


plt.figure()
#plt.plot(np.arange(epochs), P_no_mask_norms, label=r'$\|P(t)\|_F$')
plt.plot(np.arange(epochs), P_tilde_no_mask_norms, label=r'$\|P(t) X^\top\|_F$')
plt.plot(np.arange(epochs), P_tilde_no_mask_V_norms, label=r'$\|P(t) X^\top V\|_F$')
plt.plot(np.arange(epochs), P_tilde_no_mask_U_norms, label=r'$\|X^\top P(t) U\|_F$')
plt.plot(np.arange(epochs), G1_norms, label=r'$\|G_1(t)\|_F$')
plt.plot(np.arange(epochs), W2_Gamma_norms, label=r'$\|W_2(t) \Gamma_2(t)\|_F$')
plt.legend(ncol=1)
plt.title('Perturbation (without mask) and gradient norms')
plt.show()
plt.close()

plt.figure()
plt.plot(np.arange(epochs), G1_norms, label=r'$\|G_1(t)\|_F$')
plt.plot(np.arange(epochs), W2_Gamma_norms, label=r'$\|W_2^\top(t) \Gamma_2(t)\|_F$')
plt.plot(np.arange(epochs), P_norms, label=r'$\|P(t)\|_F$ with mask')
plt.plot(np.arange(epochs), P_no_mask_norms, label=r'$\|P(t)\|_F$ without mask')
plt.plot(np.arange(epochs), W2_D2_norms, label=r'$\|W_2^\top(t) \Delta_2(t)\|_F$')
plt.plot(np.arange(epochs), P_tilde_norms, label=r'$\|P(t) X^\top\|_F$ with mask')
plt.plot(np.arange(epochs), P_tilde_no_mask_norms, label=r'$\|P(t) X^\top\|_F$ without mask')
plt.plot(np.arange(epochs), phi_neg_norms, label=r'$\phi_{-}(W_1(t) X)$')
#plt.plot(np.arange(epochs), W2_Gamma_V_norms, label=r'$\|W_2^\top(t) \Gamma_2(t) V\|_F$')
#plt.plot(np.arange(epochs), W2_Gamma_U_norms, label=r'$\|\Gamma_2^\top(t) W_2(t) U\|_F$')
plt.legend(ncol=1, loc='upper right')
plt.grid('on')
plt.title('Gradient and Perturbation Norms')
plt.show()
plt.close()

plt.figure()

plt.show()


# Nullspace evolution

In [ ]:
W1_0 = weights[0][0]
W2_0 = weights[0][1]
G1_0 = grads[0][0]

E_0 = (W2_0 @ act(W1_0 @ X) - Y)

U_null, V_null = small_update_subspace_2(weights[0], E_0, X, K, init_scale)

G_norms = []

W2_Gamma_V_norms = []
Gamma_W21_V_norms = []

Vt_V_errors = []
for t in range(epochs):
    G_t = grads[t][0]
    W1_t = weights[t][0]
    W2_t = weights[t][1]
    Z2_t = W2_t @ act(W1_t @ X)
    E_t = Z2_t - Y
    
    G_norms.append(np.linalg.norm(G_t, ord='fro'))
    
    Gamma_t = E_t @ X.T
    
    W2_Gamma_t = W2_t.T @ Gamma_t
    Gamma_W21_t = Gamma_t.T @ W2_t @ W1_t
    
    W2_Gamma_V_norm = np.linalg.norm(W2_Gamma_t @ V_null, ord='fro')
    Gamma_W21_V_norm = np.linalg.norm(Gamma_W21_t @ V_null, ord='fro')
    
    W2_Gamma_V_norms.append(W2_Gamma_V_norm)
    Gamma_W21_V_norms.append(Gamma_W21_V_norm)
    
    _, Vt_null = small_update_subspace_2(weights[t], E_t, X, K, init_scale)
    Vt_V_errors.append(subspace_error(Vt_null, V_null))
    
    
    

plt.figure()
#plt.plot(np.arange(epochs) + 1, G_norms, label=r'$\| G(t) \|_F$')
plt.plot(np.arange(epochs) + 1, W2_Gamma_V_norms, label=r'$\| W_2^\top(t) \Gamma(t) V \|_F$')
plt.plot(np.arange(epochs) + 1, Gamma_W21_V_norms, label=r'$\| \Gamma^\top(t) W_{2:1}(t) V \|_F$')
plt.plot(np.arange(epochs) + 1, Vt_V_errors, label=r'$V$ and $V(t)$ subspace errors')
plt.title('Nullspace Evolutions (' + act_fn_str + ')')
plt.yscale('linear')
plt.legend()
plt.grid()
plt.show()
plt.close()





# Low-rank structure under masks

In [ ]:
step = 0.01
num_steps = np.int8(0.2 / step)
pct = [1] + [1 - i*step for i in range(1, num_steps + 1)]

# Singular values under "ReLU activation mask"
num_trials = 5
avg_U_dist = np.zeros(len(pct))
avg_V_dist = np.zeros(len(pct))

eps = 0.01

for i in range(num_trials):
    U = np.random.randn(d, r) / np.sqrt(r)
    V = np.random.randn(N, r) / np.sqrt(r)

    A = eps * (U @ V.T)
    
    U_orig, _, V_orig = np.linalg.svd(A)
    V_orig = V_orig.T
    
    for (j, p) in enumerate(pct):
        mask = np.random.rand(*A.shape) < p
        A_mask = A * mask
        
        
        U_mask, _, V_mask = np.linalg.svd(A_mask)
        V_mask = V_mask.T
        
        avg_U_dist[j] += subspace_error(U_orig[:, :r], U_mask[:, :r])
        avg_V_dist[j] += subspace_error(V_orig[:, :r], V_mask[:, :r])

avg_U_dist /= num_trials
avg_V_dist /= num_trials

# Plotting
#orig_svals = np.linalg.svd(X, compute_uv=False)
plt.figure()
plt.scatter(1 - np.array(pct), avg_U_dist, label='Left singular subspace')
plt.scatter(1 - np.array(pct), avg_V_dist, label='Right singular subspace')
plt.legend()
plt.xlabel('Percentage masked')
plt.ylabel('Subspace distance')
plt.show()
plt.close()

In [ ]:
step = 0.01
num_steps = np.int8(0.2 / step)
pct = [1] + [1 - i*step for i in range(1, num_steps + 1)]

# Singular values under "ReLU activation mask"
alpha = 0.1
num_trials = 5
avg_U_dist = np.zeros(len(pct))
avg_V_dist = np.zeros(len(pct))

eps = 0.01

for i in range(num_trials):
    U = np.random.randn(d, r) / np.sqrt(r)
    V = np.random.randn(N, r) / np.sqrt(r)

    A = eps * (U @ V.T)
    
    U_orig, _, V_orig = np.linalg.svd(A)
    V_orig = V_orig.T
    
    for (j, p) in enumerate(pct):
        mask_pos = np.random.rand(*A.shape) < p
        mask_neg = np.ones(X.shape) - mask_pos
        
        A_mask = A * mask_pos + alpha * A * mask_neg
        
        
        U_mask, _, V_mask = np.linalg.svd(A_mask)
        V_mask = V_mask.T
        
        avg_U_dist[j] += subspace_error(U_orig[:, :r], U_mask[:, :r])
        avg_V_dist[j] += subspace_error(V_orig[:, :r], V_mask[:, :r])

avg_U_dist /= num_trials
avg_V_dist /= num_trials

# Plotting
#orig_svals = np.linalg.svd(X, compute_uv=False)
plt.figure()
plt.scatter(1 - np.array(pct), avg_U_dist, label='Left singular subspace')
plt.scatter(1 - np.array(pct), avg_V_dist, label='Right singular subspace')
plt.legend()
plt.xlabel('Percentage masked')
plt.ylabel('Subspace distance')
plt.show()
plt.close()